<a href="https://colab.research.google.com/github/angadghatode/ISYS2001-Assignment2/blob/main/Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2 - Personal Finance App

## Step 1: Understand the Problem
**The Problem:** Casual workers get paid by the hour, but when they buy something, the price tag doesn't show what it actually cost them. $120 sounds like a small number, but it can be a full shift of work after tax. With shifts that change every week, it's even harder to know what you can afford without setting your savings back.

**The User:** A casual retail or hospitality worker (like a uni student) who wants to save money but has unpredictable hours.

**The Solution:** A "What Does It Really Cost?" assistant. It learns your real take-home hourly pay from your past payslips. Then you type in something you want to buy, and it tells you the cost in hours of work and how many days it pushes back your savings goal. A friendly AI coach (Gemini) then asks "still want it?" and talks it through using your real numbers.

## Step 2: Inputs and Outputs
**Inputs:**
*   Payslip data: a .csv file with the columns Pay Date, Hours Worked, Gross Pay, Tax Withheld.

*   Savings goal: a goal amount and how much the user saves per week.

*   Purchase: the item name and price, typed in by the user.
*   Chat messages: questions like "can I afford this?" or "why is my tax higher this week?"

**Outputs:**
*   Cost-in-Hours Calculator (custom tool): a Python function you write yourself that works out:

    *  Average take-home hourly pay: (Gross − Tax) ÷ Hours
    * Hours of work the purchase costs: Price ÷ take-home hourly pay
    * Days the purchase delays the savings goal: Price ÷ daily savings rate

* AI Coach (Gemini): a friendly, honest friend that uses those numbers to reply, like "That's 6 hours of work and sets your goal back 9 days. Still want it?"
* Gradio interface: one page with the CSV upload, the purchase box, the results, and the chat.

## Step 3: Worked Example
Before coding, let's calculate the real cost of a purchase manually.

**1. The Fake Payslips**
*   Week 1: 15 hours, `$450` Gross, `$50` Tax (Net: `$400`)
*   Week 2: 20 hours, `$600` Gross, `$80` Tax (Net: `$520`)

**2. The User Inputs**
*   Target Purchase: Headphones (`$350`)
*   Savings Goal: `$100` per week

**3. The Manual Math**
*   Total Hours Worked: 15 + 20 = 35 hours
*   Total Net Pay: 400 + 520 = `$920`
*   Average Net Hourly Pay: `$920` / 35 hours = `$26.28` per hour
*   Cost in Hours: `$350` / `$26.28` = 13.3 hours of actual work
*   Daily Savings Rate: `$100` / 7 days = `$14.28` per day
*   Days Delayed: `$350` / `$14.28` = 24.5 days

## Step 4 - Pseudocode

```
FUNCTION calculate_real_cost(csv_data, item_price, weekly_savings):
    SET total_net_pay = 0
    SET total_hours = 0
    
    FOR EACH row IN csv_data:
        SET net_pay = row["Gross Pay"] - row["Tax Withheld"]
        ADD net_pay TO total_net_pay
        ADD row["Hours Worked"] TO total_hours
        
    IF total_hours EQUALS 0:
        RETURN "Error: No hours logged."
        
    SET average_takehome_hourly = total_net_pay / total_hours
    SET cost_in_hours = item_price / average_takehome_hourly
    
    SET daily_savings = weekly_savings / 7
    SET days_delayed = item_price / daily_savings
    
    RETURN average_takehome_hourly, cost_in_hours, days_delayed
```



In [1]:
%%writefile payslips.csv
Pay Date,Hours Worked,Gross Pay,Tax Withheld
2026-09-15,15,450,50
2026-09-22,20,600,80

Writing payslips.csv


## Step 5 (python)

In [2]:
import pandas as pd

def calculate_real_cost(csv_path, item_price, weekly_savings):
    df = pd.read_csv(csv_path)

    df['Net Pay'] = df['Gross Pay'] - df['Tax Withheld']
    total_net_pay = df['Net Pay'].sum()
    total_hours = df['Hours Worked'].sum()

    if total_hours == 0:
        return "Error: No hours logged."

    average_takehome_hourly = total_net_pay / total_hours
    cost_in_hours = item_price / average_takehome_hourly

    daily_savings = weekly_savings / 7
    days_delayed = item_price / daily_savings

    return {
        "Hourly Rate": round(average_takehome_hourly, 2),
        "Cost in Hours": round(cost_in_hours, 1),
        "Days Delayed": round(days_delayed, 1)
    }

result = calculate_real_cost('payslips.csv', 350, 100)
print(f"Calculation Result: {result}")

Calculation Result: {'Hourly Rate': np.float64(26.29), 'Cost in Hours': np.float64(13.3), 'Days Delayed': 24.5}


In [3]:
%%writefile zero_hours_payslip.csv
Pay Date,Hours Worked,Gross Pay,Tax Withheld
2026-10-05,0,0,0

Writing zero_hours_payslip.csv


In [4]:

normal_result = calculate_real_cost('payslips.csv', 350, 100)
assert normal_result["Hourly Rate"] == 26.28 or normal_result["Hourly Rate"] == 26.29, f"Failed: Expected 26.28 or 26.29, got {normal_result['Hourly Rate']}"
assert normal_result["Cost in Hours"] == 13.3, f"Failed: Expected 13.3, got {normal_result['Cost in Hours']}"
assert normal_result["Days Delayed"] == 24.5, f"Failed: Expected 24.5, got {normal_result['Days Delayed']}"
print("Happy Path Test passed!")

error_result = calculate_real_cost('zero_hours_payslip.csv', 350, 100)
assert error_result == "Error: No hours logged.", f"Failed: Expected 'Error: No hours logged.', got {error_result}"
print("Edge Case Test passed!")

Happy Path Test passed!
Edge Case Test passed!


In [11]:
!pip install -q -U google-generativeai

import google.generativeai as genai
from google.colab import userdata

# Safely load the API key
GOOGLE_API_KEY = userdata.get('GEMINIKEY')
genai.configure(api_key=GOOGLE_API_KEY)

def ask_finance_coach(purchase_item, calculation_results, user_question):
    system_instruction = """You are a direct, pragmatic financial coach for a casual retail worker.
    Your goal is to help them understand the real cost of their purchases based on their actual take-home pay.
    Be conversational, empathetic about unpredictable shifts, but firm about the math.
    Do not give generic financial advice; strictly reference the calculated hours and days delayed provided in the prompt."""

    model = genai.GenerativeModel(
        model_name='gemini-flash-latest',
        system_instruction=system_instruction
    )

    prompt = f"""
    The user wants to buy: {purchase_item}.
    According to their parsed payslips, their real take-home pay is ${calculation_results['Hourly Rate']} per hour.
    This purchase will cost them {calculation_results['Cost in Hours']} hours of standing on the retail floor.
    It will set their savings goal back by {calculation_results['Days Delayed']} days.

    The user asks: "{user_question}"
    """

    response = model.generate_content(prompt)
    return response.text

ai_response = ask_finance_coach(
    purchase_item="Sony Headphones",
    calculation_results=normal_result,
    user_question="Is this purchase worth it, or should I wait until I get more shifts?"
)

print(ai_response)

Look, I get it. A solid pair of noise-canceling headphones sounds amazing, especially when you need to decompress after a chaotic shift. But let’s look at what this actually demands from you right now.

At your real take-home pay of **$26.29 an hour**, these headphones will cost you **13.3 hours** of standing on the retail floor. That is nearly two full shifts on your feet, dealing with customers and managers, purely to pay for this one item. 

Even bigger than the shift count: buying them right now sets your savings goal back by **24.5 days**. That’s almost an entire month of discipline and progress wiped out.

Because your shifts are casual and hours can dry up without warning, **you should wait**. 

Trading 13.3 hours of physical grind and blowing nearly a month of your savings timeline isn't worth the stress if your roster gets cut next week. Put this on pause. Once you actually pick up those extra shifts and have the cash banked without derailing your 24.5-day progress, then we ca

In [20]:
import gradio as gr
import pandas as pd
import requests
from google.colab import userdata

# 1. Setup the API Key
GOOGLE_API_KEY = userdata.get('GEMINIKEY')
GEMINI_URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3.8-flash:generateContent?key={GOOGLE_API_KEY}"

# 2. The Custom Data Tool
def process_file(file_obj, item_price, weekly_savings):
    if file_obj is None:
        return "⚠️ Please upload a CSV first.", None

    try:
        df = pd.read_csv(file_obj.name)
        df['Net Pay'] = df['Gross Pay'] - df['Tax Withheld']
        total_net_pay = df['Net Pay'].sum()
        total_hours = df['Hours Worked'].sum()

        if total_hours == 0:
            return "⚠️ Error: No hours logged in this CSV.", None

        average_takehome_hourly = total_net_pay / total_hours
        cost_in_hours = item_price / average_takehome_hourly
        daily_savings = weekly_savings / 7
        days_delayed = item_price / daily_savings

        results = {
            "Hourly Rate": round(average_takehome_hourly, 2),
            "Cost in Hours": round(cost_in_hours, 1),
            "Days Delayed": round(days_delayed, 1)
        }

        success_message = f"✅ **Data Processed!**\nYour real take-home pay is **${results['Hourly Rate']}/hr**.\nThis purchase costs **{results['Cost in Hours']} hours** of floor work and delays your savings by **{results['Days Delayed']} days**."
        return success_message, results

    except Exception as e:
        return f"⚠️ Error parsing CSV: {str(e)}", None

# 3. The AI Chat Tool
def chat_with_coach(user_message, chat_history, math_results, item_name):
    # Safeguard against empty state
    if chat_history is None:
        chat_history = []

    # Prevent empty submissions
    if not user_message.strip():
        return "", chat_history

    # Force the user to calculate the math first
    if not math_results:
        chat_history.append({"role": "user", "content": user_message})
        chat_history.append({"role": "assistant", "content": "⚠️ Please upload your CSV and click Calculate Reality before asking questions!"})
        return "", chat_history

    # Build the prompt grounded in the CSV math
    prompt = f"""
    The user wants to buy: {item_name}.
    According to their parsed payslips, their real take-home pay is ${math_results['Hourly Rate']} per hour.
    This purchase will cost them {math_results['Cost in Hours']} hours of standing on the retail floor.
    It will set their savings goal back by {math_results['Days Delayed']} days.

    The user asks: \"{user_message}\"
    """

    system_instruction = "You are a direct, pragmatic financial coach for a casual retail worker. Your goal is to help them understand the real cost of their purchases based on their actual take-home pay. Be conversational, empathetic about unpredictable shifts, but firm about the math. Do not give generic financial advice; strictly reference the calculated hours and days delayed provided in the prompt."

    payload = {
        "systemInstruction": {
            "parts": [{"text": system_instruction}]
        },
        "contents": [{
            "role": "user",
            "parts": [{"text": prompt}]
        }]
    }

    try:
        response = requests.post(GEMINI_URL, json=payload)
        response_data = response.json()

        if 'error' in response_data:
            ai_text = f"⚠️ Google API Error: {response_data['error'].get('message', 'Unknown Error')}"
        else:
            ai_text = response_data['candidates'][0]['content']['parts'][0]['text']

    except Exception as e:
        ai_text = f"⚠️ Code Error: {str(e)}"

    # Append using the strictly enforced dictionary format
    chat_history.append({"role": "user", "content": user_message})
    chat_history.append({"role": "assistant", "content": ai_text})

    return "", chat_history

# 4. The Interface Layout
with gr.Blocks() as demo:
    gr.Markdown("# 💸 The Casual Worker's Reality Check")
    gr.Markdown("Upload your payslips, set your goal, and see what your purchases really cost.")

    with gr.Row():
        with gr.Column():
            file_input = gr.File(label="Upload Payslips (CSV)", file_types=[".csv"])
            item_name = gr.Textbox(label="What do you want to buy?", value="Sony Headphones")
            item_price = gr.Number(label="How much does it cost? ($)", value=350)
            weekly_savings = gr.Number(label="Weekly Savings Target ($)", value=100)
            analyze_btn = gr.Button("Calculate Reality", variant="primary")

        with gr.Column():
            math_summary = gr.Markdown("### Upload your data and click calculate to see the math.")
            math_state = gr.State()

    gr.Markdown("---")
    gr.Markdown("### 🤖 Chat with your Financial Coach")
    chatbot = gr.Chatbot(height=300)
    chat_input = gr.Textbox(label="Ask the coach a question about your purchase")

    analyze_btn.click(
        fn=process_file,
        inputs=[file_input, item_price, weekly_savings],
        outputs=[math_summary, math_state]
    )

    chat_input.submit(
        fn=chat_with_coach,
        inputs=[chat_input, chatbot, math_state, item_name],
        outputs=[chat_input, chatbot]
    )

demo.launch(theme=gr.themes.Soft(), debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c8229a0c02f115f151.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7862 <> https://c8229a0c02f115f151.gradio.live
